# Gold Dimensional Model

## Purpose

Build business-ready dimesnions and facts from validated silver tables.

## Inputs

Validated Silver Delta Tables

## Outputs

### Dimesions

- `dim_date`
- `dim_customer`
- `dim_product`
- `dim_seller`

### Facts

- `fact_order`
- `fact_order_item`
- `fact_payment`

## Modelling Principles

- Explicit table grains
- Deterministic surrogate keys
- Star-schema relationships
- Separate fact-table grains
- No item-payment multiplication
- Complete reconciliation to silver

In [1]:
import uuid
import builtins
import time
from datetime import datetime, timezone
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 3, Finished, Available, Finished, False)

In [2]:
GOLD_RUN_ID = str(uuid.uuid4())
GOLD_STARTED_AT = datetime.now(timezone.utc)
GOLD_START_PREF = time.perf_counter()

LOAD_TYPE = "FULL"

print("Gold run ID:", GOLD_RUN_ID)
print("Started at:", GOLD_STARTED_AT)
print("Load type:", LOAD_TYPE)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 4, Finished, Available, Finished, False)

Gold run ID: 476235ad-0021-48a2-a3e1-706713719f79
Started at: 2026-07-31 17:17:25.217986+00:00
Load type: FULL


# Gold helper functions

Reusable helpers add gold metadata, write delta tables and record dimension-load audits.

In [3]:
def add_gold_metadata(dataframe: DataFrame, source_tables: str) -> DataFrame:
    """ Add techincal lineage metadata to a gold dataframe """

    return (dataframe
    .withColumn("_gold_run_id", F.lit(GOLD_RUN_ID))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .withColumn("_gold_source_tables", F.lit(source_tables))
    .withColumn("_gold_load_type", F.lit(LOAD_TYPE))
    )

def write_gold_table(dataframe: DataFrame, table_name: str) -> None:
    """ Write a managed gold delta table using full overwrite. """

    dataframe.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(table_name)



StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 5, Finished, Available, Finished, False)

In [4]:
gold_dimension_audit_records = []

def record_dimension_audit(
    *,
    dimension_table: str,
    source_tables: str,
    source_entity_count: int,
    target_row_count: int,
    distinct_natural_key_count: int,
    distinct_surrogate_key_count: int,
    duplicate_natural_key_count: int,
    null_surrogate_key_count: int,
    status: str
) -> None:
    gold_dimension_audit_records.append({
        "gold_run_id" : GOLD_RUN_ID,
        "dimension_table" : dimension_table,
        "source_tables" : source_tables,
        "source_entity_count" : int(source_entity_count),
        "target_row_count" : int(target_row_count),
        "distinct_natural_key_count" : int(distinct_natural_key_count),
        "distinct_surrogate_key_count" : int(distinct_surrogate_key_count),
        "distinct_natural_key_count" : int(distinct_natural_key_count),
        "null_surrogate_key_count" : int(null_surrogate_key_count),
        "load_status" : status,
        "processed_at_utc" : datetime.now(timezone.utc)
    })

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 6, Finished, Available, Finished, False)

# Date dimension

Create a new row per calendar date across the complete analytical date range found in silver table. 

In [5]:
silver_orders = spark.table("silver_orders")
silver_order_reviews = spark.table("silver_order_reviews")
silver_order_items = spark.table("silver_order_items")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 7, Finished, Available, Finished, False)

In [6]:
order_date_bounds = (
    silver_orders
    .select(
        F.min(
            F.least(
                F.to_date("order_purchase_ts"),
                F.to_date("order_approved_ts"),
                F.to_date("carrier_handover_ts"),
                F.to_date("delivered_customer_ts"),
                F.to_date("estimated_delivery_ts")
            )
        ).alias("minimum_date"),
        F.max(
            F.greatest(
                F.to_date("order_purchase_ts"),
                F.to_date("order_approved_ts"),
                F.to_date("carrier_handover_ts"),
                F.to_date("delivered_customer_ts"),
                F.to_date("estimated_delivery_ts")
            )
        ).alias("maximum_date")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 8, Finished, Available, Finished, False)

In [7]:
date_candidates = (
    silver_orders
    .select(F.to_date("order_purchase_ts").alias("calendar_date"))
    .unionByName(silver_orders.select(F.to_date("order_approved_ts").alias("calendar_date")))
    .unionByName(silver_orders.select(F.to_date("carrier_handover_ts").alias("calendar_date")))
    .unionByName(silver_orders.select(F.to_date("delivered_customer_ts").alias("calendar_date")))
    .unionByName(silver_orders.select(F.to_date( "estimated_delivery_ts").alias("calendar_date")))
    .unionByName(silver_order_items.select(F.to_date("shipping_limit_ts").alias("calendar_date")))
    .unionByName(silver_order_reviews.select(F.to_date("review_created_ts").alias("calendar_date")))
    .unionByName(silver_order_reviews.select(F.to_date("review_answered_ts").alias("calendar_date")))
    .filter(F.col("calendar_date").isNotNull())
)

date_bounds = (
    date_candidates
    .agg(
        F.min("calendar_date").alias("minimum_date"),
        F.max("calendar_date").alias("maximum_date")
    )
    .collect()[0]
)

MINIMUM_DATE = date_bounds["minimum_date"]
MAXIMUM_DATE = date_bounds["maximum_date"]

print("Minimum date:", MINIMUM_DATE)
print("Maximum date:", MAXIMUM_DATE)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 9, Finished, Available, Finished, False)

Minimum date: 2016-09-04
Maximum date: 2020-04-09


In [8]:
dim_date = spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{MINIMUM_DATE}'),
            to_date('{MAXIMUM_DATE}'),
            interval 1 day
        )
    ) AS date
    """
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 10, Finished, Available, Finished, False)

In [9]:
dim_date = (
    dim_date
    .withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter_number", F.quarter("date"))
    .withColumn("quarter_name", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("year_month_sort", F.date_format("date", "yyyyMM").cast("int"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_year", F.dayofyear("date"))
    .withColumn("day_of_week_number", F.dayofweek("date"))
    .withColumn("day_of_week_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 11, Finished, Available, Finished, False)

In [10]:
dim_date = add_gold_metadata(dim_date, "silver_orders, silver_order_items, silver_order_reviews")

write_gold_table(dim_date, "dim_date")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 12, Finished, Available, Finished, False)

In [11]:
dim_date_saved = spark.table("dim_date")

dim_date_row_count = dim_date_saved.count()
expected_date_count = (MAXIMUM_DATE - MINIMUM_DATE).days + 1
duplicate_date_count = dim_date_saved.groupBy("date").count().filter(F.col("count") > 1).count()
duplicate_date_key_count = dim_date_saved.groupBy("date_key").count().filter(F.col("count") > 1).count()
null_date_key_count = dim_date_saved.filter(F.col("date_key").isNull()).count()

dim_date_status = (
    "SUCCESS"
    if (
        dim_date_row_count == expected_date_count 
        and duplicate_date_count == 0 
        and duplicate_date_key_count == 0
        and null_date_key_count == 0
        )
    else "FAILED"
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 13, Finished, Available, Finished, False)

In [12]:
record_dimension_audit(
    dimension_table = "dim_table",
    source_tables = "silver_orders, silver_order_items, silver_order_reviews",
    source_entity_count = expected_date_count,
    target_row_count = dim_date_row_count,
    distinct_natural_key_count = dim_date_saved.select("date").distinct().count(),
    distinct_surrogate_key_count = dim_date_saved.select("date_key").distinct().count(),
    duplicate_natural_key_count = duplicate_date_count,
    null_surrogate_key_count = null_date_key_count,
    status = dim_date_status
)

display(dim_date_saved.orderBy("date").limit(20))

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8be5905d-b256-43c2-bd61-0e46496e5249)

# Customer Dimension

Create one row per customer record and enrich it with persistent-customer history and ZIP-level coordinates.

In [13]:
silver_customers = spark.table("silver_customers")
silver_orders = spark.table("silver_orders")
silver_geolocation_zip = spark.table("silver_geolocation_zip")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 15, Finished, Available, Finished, False)

In [14]:
customer_order_history = (
    silver_orders.alias("o")
    .join(silver_customers.select("customer_id","customer_unique_id").alias("c"),
    on = "customer_id",
    how = "left")
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 16, Finished, Available, Finished, False)

In [15]:
customer_unique_summary = (
    customer_order_history
    .groupBy("customer_unique_id")
    .agg(
        F.min("order_purchase_ts").alias("first_order_ts"),
        F.max("order_purchase_ts").alias("last_order_ts"),
        F.countDistinct("order_id").alias("lifetime_order_count")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 17, Finished, Available, Finished, False)

In [16]:
dim_customer_base = (
    silver_customers.alias("c")
    .join(
        customer_unique_summary.alias("h"),
        on="customer_unique_id",
        how="left"
    )
    .join(
        silver_geolocation_zip
        .select(
            F.col("zip_prefix").alias("customer_zip_prefix"),
            F.col("average_latitude").alias("customer_latitude"),
            F.col("average_longitude").alias("customer_longitude"),
            F.col("primary_city").alias("zip_primary_city"),
            F.col("primary_state").alias("zip_primary_state")
        ).alias("g"),
        on="customer_zip_prefix",
        how="left"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 18, Finished, Available, Finished, False)

In [17]:
dim_customer_base = (
    dim_customer_base
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_prefix",
        "customer_city",
        "customer_state",
        "customer_latitude",
        "customer_longitude",
        "zip_primary_city",
        "zip_primary_state",
        F.to_date("first_order_ts").alias("first_order_date"),
        F.to_date("last_order_ts").alias("last_order_date"),
        F.coalesce(F.col("lifetime_order_count"), F.lit(0)).cast("long").alias("lifetime_order_count"),
        F.when(F.col("lifetime_order_count") > 1, F.lit("Repeat")).otherwise(F.lit("New")).alias("customer_type"),
        F.col("customer_latitude").isNull().alias("_dq_missing_customer_coordinates")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 19, Finished, Available, Finished, False)

In [18]:
customer_key_window = Window.orderBy("customer_id")

dim_customer = dim_customer_base.withColumn("customer_key", F.row_number().over(customer_key_window).cast("long"))

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 20, Finished, Available, Finished, False)

In [19]:
dim_customer = (
    dim_customer
    .select(
        "customer_key",
        "customer_id",
        "customer_unique_id",
        "customer_zip_prefix",
        "customer_city",
        "customer_state",
        "customer_latitude",
        "customer_longitude",
        "zip_primary_city",
        "zip_primary_state",
        "first_order_date",
        "last_order_date",
        "lifetime_order_count",
        "customer_type",
        "_dq_missing_customer_coordinates"
    )
)

dim_customer = add_gold_metadata(dim_customer, "silver_customers, silver_orders, silver_geolocation_zip")

write_gold_table(dim_customer, "dim_customer")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 21, Finished, Available, Finished, False)

In [20]:
dim_customer_saved = spark.table("dim_customer")

customer_source_count = silver_customers.count()
customer_target_count = dim_customer_saved.count()
duplicate_customer_id_count = dim_customer_saved.groupBy("customer_id").count().filter(F.col("count") > 1).count()
duplicate_customer_key_count = dim_customer_saved.groupBy("customer_key").count().filter(F.col("count") > 1).count()
null_customer_key_count = dim_customer_saved.filter(F.col("customer_key").isNull()).count()
dim_customer_status = (
    "SUCCESS"
    if (
        customer_source_count == customer_target_count
        and duplicate_customer_id_count == 0
        and duplicate_customer_key_count == 0
        and null_customer_key_count == 0
    )
    else "FAILED"
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 22, Finished, Available, Finished, False)

In [21]:
record_dimension_audit(
    dimension_table = "dim_customer",
    source_tables = "silver_customers, silver_orders, silver_geolocation_zip",
    source_entity_count = customer_source_count,
    target_row_count = customer_target_count,
    distinct_natural_key_count = dim_customer_saved.select("customer_id").distinct().count(),
    distinct_surrogate_key_count = dim_customer_saved.select("customer_key").distinct().count(),
    duplicate_natural_key_count = duplicate_customer_id_count,
    null_surrogate_key_count = null_customer_key_count,
    status = dim_customer_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 23, Finished, Available, Finished, False)

In [22]:
display(
    dim_customer_saved
    .select(
        "customer_key",
        "customer_id",
        "customer_unique_id",
        "customer_state",
        "customer_type",
        "lifetime_order_count",
        "first_order_date",
        "last_order_date"
    )
    .orderBy(F.col("lifetime_order_count").desc(),"customer_key")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 305d67b5-9a31-45e3-b8af-5dcaff2a3b38)

# Product Dimension

Create one row per product with translated category and physical attributes.sil

In [23]:
silver_products = spark.table("silver_products")

dim_product_base = (
    silver_products
    .select(
        "product_id",
        "product_category_name_pt",
        "product_category_name_en",
        "product_name_length",
        "product_description_length",
        "product_photo_count",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
        "product_density_g_per_cm3",
        F.when(F.col("_dq_missing_category_translation"),F.lit("Missing")).otherwise(F.lit("Available")).alias("category_translation_status"),
        "_dq_missing_category",
        "_dq_invalid_weight",
        "_dq_invalid_dimensions"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 25, Finished, Available, Finished, False)

In [24]:
product_key_window = Window.orderBy("product_id")

dim_product = dim_product_base.withColumn("product_key", F.row_number().over(product_key_window).cast("long"))

dim_product = (
    dim_product
    .select(
        "product_key",
        "product_id",
        "product_category_name_pt",
        "product_category_name_en",
        "category_translation_status",
        "product_name_length",
        "product_description_length",
        "product_photo_count",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
        "product_density_g_per_cm3",
        "_dq_missing_category",
        "_dq_invalid_weight",
        "_dq_invalid_dimensions"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 26, Finished, Available, Finished, False)

In [25]:
dim_product = add_gold_metadata(dim_product, "silver_products")

write_gold_table(dim_product, "dim_product")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 27, Finished, Available, Finished, False)

In [26]:
dim_product_saved = spark.table("dim_product")

product_source_count = silver_products.count()
product_target_count = dim_product_saved.count()

duplicate_product_id_count = (
    dim_product_saved
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_product_key_count = (
    dim_product_saved
    .groupBy("product_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_product_key_count = (
    dim_product_saved
    .filter(F.col("product_key").isNull())
    .count()
)

dim_product_status = (
    "SUCCESS"
    if (
        product_source_count == product_target_count
        and duplicate_product_id_count == 0
        and duplicate_product_key_count == 0
        and null_product_key_count == 0
    )
    else "FAILED"
)

record_dimension_audit(
    dimension_table = "dim_product",
    source_tables = "silver_products",
    source_entity_count = product_source_count,
    target_row_count = product_target_count,
    distinct_natural_key_count = dim_product_saved.select("product_id").distinct().count(),
    distinct_surrogate_key_count = dim_product_saved.select("product_key").distinct().count(),
    duplicate_natural_key_count = duplicate_product_id_count,
    null_surrogate_key_count = null_product_key_count,
    status = dim_product_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 28, Finished, Available, Finished, False)

In [27]:
display(
    dim_product_saved
    .select(
        "product_key",
        "product_id",
        "product_category_name_en",
        "category_translation_status",
        "product_weight_g",
        "product_volume_cm3"
    )
    .orderBy("product_key")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 658f9131-9dde-4aa3-877c-fdd9a36a8e9c)

# Seller Dimension

Create one row per seller and enrich it with ZIP-level coordinates.

In [28]:
silver_sellers = spark.table("silver_sellers")
silver_geolocation_zip = spark.table("silver_geolocation_zip")

dim_seller_base = (
    silver_sellers.alias("s")
    .join(
        silver_geolocation_zip
        .select(
            F.col("zip_prefix").alias("seller_zip_prefix"),
            F.col("average_latitude").alias("seller_latitude"),
            F.col("average_longitude").alias("seller_longitude"),
            F.col("primary_city").alias("zip_primary_city"),
            F.col("primary_state").alias("zip_primary_state")
        )
        .alias("g"),
        on="seller_zip_prefix",
        how="left"
    )
    .select(
        "seller_id",
        "seller_zip_prefix",
        "seller_city",
        "seller_state",
        "seller_latitude",
        "seller_longitude",
        "zip_primary_city",
        "zip_primary_state",
        F.col("seller_latitude").isNull().alias("_dq_missing_seller_coordinates")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 30, Finished, Available, Finished, False)

In [30]:
seller_key_window = Window.orderBy("seller_id")

dim_seller = (
    dim_seller_base
    .withColumn("seller_key",F.row_number().over(seller_key_window).cast("long"))
)

dim_seller = (
    dim_seller
    .select(
        "seller_key",
        "seller_id",
        "seller_zip_prefix",
        "seller_city",
        "seller_state",
        "seller_latitude",
        "seller_longitude",
        "zip_primary_city",
        "zip_primary_state",
        "_dq_missing_seller_coordinates"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 32, Finished, Available, Finished, False)

In [31]:
dim_seller = add_gold_metadata(dim_seller, "silver_sellers, silver_geolocation_zip")

write_gold_table(dim_seller,"dim_seller")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 33, Finished, Available, Finished, False)

In [32]:
dim_seller_saved = spark.table("dim_seller")

seller_source_count = silver_sellers.count()
seller_target_count = dim_seller_saved.count()

duplicate_seller_id_count = (
    dim_seller_saved
    .groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_seller_key_count = (
    dim_seller_saved
    .groupBy("seller_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_seller_key_count = dim_seller_saved.filter(F.col("seller_key").isNull()).count()

dim_seller_status = (
    "SUCCESS"
    if (
        seller_source_count == seller_target_count
        and duplicate_seller_id_count == 0
        and duplicate_seller_key_count == 0
        and null_seller_key_count == 0
    )
    else "FAILED"
)

record_dimension_audit(
    dimension_table = "dim_seller",
    source_tables = "silver_sellers, silver_geolocation_zip",
    source_entity_count = seller_source_count,
    target_row_count = seller_target_count,
    distinct_natural_key_count = dim_seller_saved.select("seller_id").distinct().count(),
    distinct_surrogate_key_count = dim_seller_saved.select("seller_key").distinct().count(),
    duplicate_natural_key_count = duplicate_seller_id_count,
    null_surrogate_key_count = null_seller_key_count,
    status = dim_seller_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 34, Finished, Available, Finished, False)

In [33]:
display(
    dim_seller_saved
    .select(
        "seller_key",
        "seller_id",
        "seller_state",
        "seller_city",
        "seller_latitude",
        "seller_longitude",
        "_dq_missing_seller_coordinates"
    )
    .orderBy("seller_key")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 677ed5ac-aec8-47a5-be3a-eb496a725c14)

In [34]:
### Validate all dimensions

EXPECTED_DIMENSIONS = {"dim_date", "dim_customer", "dim_product", "dim_seller"}

missing_dimensions = {
    table_name
    for table_name in EXPECTED_DIMENSIONS
    if not spark.catalog.tableExists(table_name)
}

if missing_dimensions:
    raise RuntimeError(
        f"Missing Gold dimensions: "
        f"{missing_dimensions}"
    )

print("All Gold dimensions exist.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 36, Finished, Available, Finished, False)

All Gold dimensions exist.


In [35]:
failed_dimension_loads = [
    record
    for record in gold_dimension_audit_records
    if record["load_status"] != "SUCCESS"
]

if failed_dimension_loads:
    raise RuntimeError(
        "Gold dimension validation failed: "
        f"{failed_dimension_loads}"
    )

print("All four Gold dimensions validated successfully.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 37, Finished, Available, Finished, False)

All four Gold dimensions validated successfully.


In [39]:
gold_dimension_audit_schema = StructType([
    StructField("gold_run_id", StringType(), False),
    StructField("dimension_table", StringType(), False),
    StructField("source_tables", StringType(), False),
    StructField("source_entity_count", LongType(), False),
    StructField("target_row_count", LongType(), False),
    StructField("distinct_natural_key_count", LongType(), False),
    StructField("distinct_surrogate_key_count", LongType(), False),
    StructField("duplicate_natural_key_count", LongType(), False),
    StructField("null_surrogate_key_count", LongType(), False),
    StructField("load_status", StringType(), False),
    StructField("processed_at_utc", TimestampType(), False)
])

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 41, Finished, Available, Finished, False)

In [45]:
gold_dimension_audit_records[0]["dimension_table"] = "dim_date"

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 47, Finished, Available, Finished, False)

In [42]:
print("gold_dimension_audit_records:", type(gold_dimension_audit_records))
print("gold_dimension_audit_schema:", type(gold_dimension_audit_schema))

print("Records is None:", gold_dimension_audit_records is None)

print("Schema is None:", gold_dimension_audit_schema is None)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 44, Finished, Available, Finished, False)

gold_dimension_audit_records: <class 'list'>
gold_dimension_audit_schema: <class 'pyspark.sql.types.StructType'>
Records is None: False
Schema is None: False


In [43]:
print(gold_dimension_audit_records)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 45, Finished, Available, Finished, False)

[{'gold_run_id': '476235ad-0021-48a2-a3e1-706713719f79', 'dimension_table': 'dim_table', 'source_tables': 'silver_orders, silver_order_items, silver_order_reviews', 'source_entity_count': 1314, 'target_row_count': 1314, 'distinct_natural_key_count': 1314, 'distinct_surrogate_key_count': 1314, 'null_surrogate_key_count': 0, 'load_status': 'SUCCESS', 'processed_at_utc': datetime.datetime(2026, 7, 31, 17, 18, 21, 21664, tzinfo=datetime.timezone.utc)}, {'gold_run_id': '476235ad-0021-48a2-a3e1-706713719f79', 'dimension_table': 'dim_customer', 'source_tables': 'silver_customers, silver_orders, silver_geolocation_zip', 'source_entity_count': 99441, 'target_row_count': 99441, 'distinct_natural_key_count': 99441, 'distinct_surrogate_key_count': 99441, 'null_surrogate_key_count': 0, 'load_status': 'SUCCESS', 'processed_at_utc': datetime.datetime(2026, 7, 31, 17, 18, 59, 893309, tzinfo=datetime.timezone.utc)}, {'gold_run_id': '476235ad-0021-48a2-a3e1-706713719f79', 'dimension_table': 'dim_product

In [44]:
print(gold_dimension_audit_schema)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 46, Finished, Available, Finished, False)

StructType([StructField('gold_run_id', StringType(), False), StructField('dimension_table', StringType(), False), StructField('source_tables', StringType(), False), StructField('source_entity_count', LongType(), False), StructField('target_row_count', LongType(), False), StructField('distinct_natural_key_count', LongType(), False), StructField('distinct_surrogate_key_count', LongType(), False), StructField('duplicate_natural_key_count', LongType(), False), StructField('null_surrogate_key_count', LongType(), False), StructField('load_status', StringType(), False), StructField('processed_at_utc', TimestampType(), False)])


In [49]:
for record in gold_dimension_audit_records:
    record["duplicate_natural_key_count"] = (
        record["target_row_count"] - record["distinct_natural_key_count"]
    )

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 51, Finished, Available, Finished, False)

In [52]:
gold_dimension_audit_df = spark.createDataFrame(
    gold_dimension_audit_records, schema=gold_dimension_audit_schema
)

gold_dimension_audit_df.write.mode("append").format("delta").saveAsTable("audit_gold_dimension_load")
    
display(gold_dimension_audit_df)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 27d5ad1f-9acd-41d6-9353-4d2f193225aa)

In [53]:
display(
    spark.table("dim_date")
    .select(
        "date_key",
        "date",
        "year",
        "quarter_name",
        "month_number",
        "month_name",
        "year_month",
        "week_of_year",
        "day_of_week_name",
        "is_weekend"
    )
    .orderBy("date")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d2daea5-9d8c-4de6-bb02-20288a0745f8)

In [54]:
display(
    spark.table("dim_customer")
    .select(
        "customer_key",
        "customer_id",
        "customer_unique_id",
        "customer_state",
        "customer_city",
        "customer_type",
        "lifetime_order_count",
        "first_order_date",
        "last_order_date",
        "customer_latitude",
        "customer_longitude"
    )
    .orderBy(
        F.col("lifetime_order_count").desc(),
        "customer_key"
    )
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f8031fee-6d43-49c2-a4cb-62c54b1ff6cf)

In [55]:
display(
    spark.table("dim_product")
    .select(
        "product_key",
        "product_id",
        "product_category_name_pt",
        "product_category_name_en",
        "category_translation_status",
        "product_photo_count",
        "product_weight_g",
        "product_volume_cm3",
        "product_density_g_per_cm3"
    )
    .orderBy("product_key")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a11d37b-84f9-4d99-9821-2ab598e0bd23)

In [56]:
display(
    spark.table("dim_seller")
    .select(
        "seller_key",
        "seller_id",
        "seller_state",
        "seller_city",
        "seller_zip_prefix",
        "seller_latitude",
        "seller_longitude",
        "_dq_missing_seller_coordinates"
    )
    .orderBy("seller_key")
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9787623e-2b8c-4e72-b211-69b9436fc367)

In [57]:
display(
    spark.table("audit_gold_dimension_load")
    .filter(
        F.col("gold_run_id") == GOLD_RUN_ID
    )
    .select(
        "dimension_table",
        "source_entity_count",
        "target_row_count",
        "distinct_natural_key_count",
        "distinct_surrogate_key_count",
        "duplicate_natural_key_count",
        "null_surrogate_key_count",
        "load_status"
    )
    .orderBy("dimension_table")
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c91932f7-d688-4a1e-ad66-65783305dff9)

# Gold Fact Implementation

This section creates the Gold fact tables at their documented business grains.

Facts created:

- `fact_order`: one row per order;
- `fact_order_item`: one row per order and item sequence;
- `fact_payment`: one row per order and payment sequence.

Order-item and payment facts remain separate to prevent cross-grain record multiplication.

In [58]:
silver_orders = spark.table("silver_orders")
silver_order_items = spark.table("silver_order_items")
silver_order_payments = spark.table("silver_order_payments")
silver_order_reviews = spark.table("silver_order_reviews")

dim_date = spark.table("dim_date")
dim_customer = spark.table("dim_customer")
dim_product = spark.table("dim_product")
dim_seller = spark.table("dim_seller")

print("Required Silver and dimension tables loaded.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 60, Finished, Available, Finished, False)

Required Silver and dimension tables loaded.


In [59]:
customer_lookup = dim_customer.select("customer_key", "customer_id")
product_lookup = dim_product.select("product_key", "product_id")
seller_lookup = dim_seller.select("seller_key", "seller_id")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 61, Finished, Available, Finished, False)

# Fact Load Audit

Collect row-count, grain, surrogate-key and load-status information for every Gold fact table.

In [61]:
gold_fact_audit_records = []

def record_fact_audit(
    *,
    fact_table: str,
    grain_description: str,
    source_tables: str,
    expected_row_count: int,
    target_row_count: int,
    distinct_grain_count: int,
    duplicate_grain_count: int,
    null_dimension_key_count: int,
    status: str
) -> None:

    gold_fact_audit_records.append({
        "gold_run_id": GOLD_RUN_ID,
        "fact_table": fact_table,
        "grain_description": grain_description,
        "source_tables": source_tables,
        "expected_row_count": int(expected_row_count),
        "target_row_count": int(target_row_count),
        "distinct_grain_count": int(distinct_grain_count),
        "duplicate_grain_count": int(duplicate_grain_count),
        "null_dimension_key_count": int(null_dimension_key_count),
        "row_count_difference": int(target_row_count - expected_row_count),
        "load_status": status,
        "processed_at_utc": datetime.now(timezone.utc)
    })

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 63, Finished, Available, Finished, False)

# Order-Level Item Aggregation

Aggregate item-level commercial values to one row per order before joining them to the order fact.

In [62]:
order_item_aggregate = (
    silver_order_items
    .groupBy("order_id")
    .agg(
        F.sum("item_price").alias("order_gmv"),
        F.sum("freight_value").alias("order_freight_value"),
        F.sum("item_total_value").alias("order_total_value"),
        F.count("*").alias("item_count"),
        F.countDistinct("product_id").alias("distinct_product_count"),
        F.countDistinct("seller_id").alias("distinct_seller_count"),
        F.avg("item_price").alias("average_item_price"),
        F.avg("freight_value").alias("average_item_freight")
    )
)

duplicate_order_item_aggregate = order_item_aggregate.groupBy("order_id").count().filter(F.col("count") > 1).count()

if duplicate_order_item_aggregate > 0:
    raise RuntimeError("Order-item aggregation is not unique by order_id.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 64, Finished, Available, Finished, False)

# Order-Level Payment Aggregation

Aggregate payment records independently to one row per order.

In [63]:
order_payment_aggregate = (
    silver_order_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("order_payment_value"),
        F.count("*").alias("payment_record_count"),
        F.max("payment_installments").alias("maximum_payment_installments"),
        F.countDistinct("payment_type").alias("distinct_payment_type_count"),
        F.max(F.when(F.col("uses_installments"), 1).otherwise(0)).cast("boolean").alias("uses_installments")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 65, Finished, Available, Finished, False)

In [64]:
review_order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("review_answered_ts").desc_nulls_last(),
        F.col("review_created_ts").desc_nulls_last(),
        F.col("review_id").asc()
    )
)

reviews_ranked_by_order = (
    silver_order_reviews
    .withColumn("_review_order_rank", F.row_number().over(review_order_window))
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 66, Finished, Available, Finished, False)

In [65]:
latest_review_lookup = (
    reviews_ranked_by_order
    .filter(F.col("_review_order_rank") == 1)
    .select(
        "order_id",
        F.col("review_id").alias("latest_review_id"),
        F.col("review_score").alias("latest_review_score"),
        F.col("review_score_group").alias("latest_review_score_group"),
        F.col("review_created_ts").alias("latest_review_created_ts"),
        F.col("review_answered_ts").alias("latest_review_answered_ts")
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 67, Finished, Available, Finished, False)

In [66]:
review_order_aggregate = (
    silver_order_reviews
    .groupBy("order_id")
    .agg(
        F.count("*").alias("review_count"),
        F.avg("review_score").alias("average_review_score"),
        F.max(F.when(F.col("has_review_comment"), 1).otherwise(0)).cast("boolean").alias("has_review_comment"),
        F.avg("review_response_hours").alias("average_review_response_hours")
    )
    .join(
        latest_review_lookup,
        on="order_id",
        how="left"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 68, Finished, Available, Finished, False)

# Order Fact

Create one row per order with customer, lifecycle, commercial, payment and review measures.

In [67]:
fact_order_base = (
    silver_orders.alias("o")
    .join(
        customer_lookup.alias("c"),
        on="customer_id",
        how="left"
    )
    .join(
        order_item_aggregate.alias("i"),
        on="order_id",
        how="left"
    )
    .join(
        order_payment_aggregate.alias("p"),
        on="order_id",
        how="left"
    )
    .join(
        review_order_aggregate.alias("r"),
        on="order_id",
        how="left"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 69, Finished, Available, Finished, False)

In [68]:
fact_order = (
    fact_order_base
    .withColumn(
        "order_date_key",
        F.date_format(F.to_date("order_purchase_ts"),"yyyyMMdd").cast("int")
    )
    .withColumn(
        "approved_date_key",
        F.when(
            F.col("order_approved_ts").isNotNull(),
            F.date_format(F.to_date("order_approved_ts"),"yyyyMMdd").cast("int")
        )
    )
    .withColumn(
        "carrier_handover_date_key",
        F.when(
            F.col("carrier_handover_ts").isNotNull(),
            F.date_format(F.to_date("carrier_handover_ts"),"yyyyMMdd").cast("int")
        )
    )
    .withColumn(
        "delivered_date_key",
        F.when(
            F.col("delivered_customer_ts").isNotNull(),
            F.date_format(F.to_date("delivered_customer_ts"),"yyyyMMdd").cast("int")
        )
    )
    .withColumn(
        "estimated_delivery_date_key",
        F.when(
            F.col("estimated_delivery_ts").isNotNull(),
            F.date_format(F.to_date("estimated_delivery_ts"),"yyyyMMdd").cast("int")
        )
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 70, Finished, Available, Finished, False)

In [69]:
fact_order = (
    fact_order
    .withColumn("order_gmv", F.coalesce(F.col("order_gmv"), F.lit(0).cast("decimal(18,2)")))
    .withColumn("order_freight_value",F.coalesce(F.col("order_freight_value"),F.lit(0).cast("decimal(18,2)")))
    .withColumn("order_total_value",F.coalesce(F.col("order_total_value"),F.lit(0).cast("decimal(18,2)")))
    .withColumn("item_count",F.coalesce(F.col("item_count"),F.lit(0)).cast("long"))
    .withColumn("distinct_product_count",F.coalesce(F.col("distinct_product_count"),F.lit(0)).cast("long"))
    .withColumn("distinct_seller_count",F.coalesce(F.col("distinct_seller_count"),F.lit(0)).cast("long"))
    .withColumn("order_payment_value",F.coalesce(F.col("order_payment_value"),F.lit(0).cast("decimal(18,2)")))
    .withColumn("payment_record_count",F.coalesce(F.col("payment_record_count"),F.lit(0)).cast("long"))
    .withColumn("review_count",F.coalesce(F.col("review_count"),F.lit(0)).cast("long"))
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 71, Finished, Available, Finished, False)

In [79]:
fact_order = (
    fact_order
    .withColumn(
        "has_order_items",
        F.col("item_count") > 0
    )
    .withColumn(
        "has_payment",
        F.col("payment_record_count") > 0
    )
    .withColumn(
        "has_review",
        F.col("review_count") > 0
    )
    .withColumn(
        "freight_share_of_order_value",
        F.when(
            F.col("order_total_value") > 0,
            (
                F.col("order_freight_value")
                / F.col("order_total_value")
            ).cast("decimal(18,6)")
        )
    )
    .withColumn(
        "payment_reconciliation_difference",
        (
            F.col("order_payment_value")
            - F.col("order_total_value")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "is_delivered",
        F.col("order_status") == "delivered"
    )
    .withColumn(
        "is_cancelled",
        F.col("order_status") == "canceled"
    )
    .withColumn(
        "is_unavailable",
        F.col("order_status") == "unavailable"
    )
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 81, Finished, Available, Finished, False)

In [80]:
print(fact_order.columns)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 82, Finished, Available, Finished, False)

['order_id', 'customer_id', 'order_status', 'order_purchase_ts', 'order_approved_ts', 'carrier_handover_ts', 'delivered_customer_ts', 'estimated_delivery_ts', '_bronze_record_hash', '_bronze_run_id', '_bronze_ingested_at', '_dq_missing_order_id', '_dq_missing_customer_id', '_dq_invalid_order_status', '_dq_missing_purchase_timestamp', '_dq_approval_before_purchase', '_dq_carrier_before_approval', '_dq_delivery_before_purchase', '_dq_delivered_status_missing_delivery_ts', '_quarantine_reason', 'order_purchase_date', 'purchase_year', 'purchase_month', 'purchase_hour', 'purchase_weekday_number', 'purchase_weekday_name', 'is_weekend_purchase', 'approval_hours', 'carrier_handover_days', 'delivery_days', 'delivery_delay_days', 'is_late_delivery', '_silver_run_id', '_silver_processed_at', '_silver_source_table', '_silver_load_type', 'customer_key', 'order_gmv', 'order_freight_value', 'order_total_value', 'item_count', 'distinct_product_count', 'distinct_seller_count', 'average_item_price', 'av

In [86]:
fact_order = (
    fact_order
    .select(
        "order_id",
        "customer_key",
        "customer_id",
        "order_date_key",
        "approved_date_key",
        "carrier_handover_date_key",
        "delivered_date_key",
        "estimated_delivery_date_key",
        "order_status",
        "order_purchase_ts",
        "order_approved_ts",
        "carrier_handover_ts",
        "delivered_customer_ts",
        "estimated_delivery_ts",
        "purchase_hour",
        "purchase_weekday_name",
        "is_weekend_purchase",
        "is_delivered",
        "is_cancelled",
        "is_unavailable",
        "is_late_delivery",
        "approval_hours",
        "carrier_handover_days",
        "delivery_days",
        "delivery_delay_days",
        "order_gmv",
        "order_freight_value",
        "order_total_value",
        "item_count",
        "distinct_product_count",
        "distinct_seller_count",
        "average_item_price",
        "average_item_freight",
        "order_payment_value",
        "payment_record_count",
        "maximum_payment_installments",
        "distinct_payment_type_count",
        "uses_installments",
        "payment_reconciliation_difference",
        "review_count",
        "average_review_score",
        "latest_review_id",
        "latest_review_score",
        "latest_review_score_group",
        "latest_review_created_ts",
        "latest_review_answered_ts",
        "has_review_comment",
        "average_review_response_hours",
        "has_order_items",
        "has_payment",
        "has_review",
        "freight_share_of_order_value",
        "_dq_approval_before_purchase",
        "_dq_carrier_before_approval",
        "_dq_delivery_before_purchase",
        "_dq_delivered_status_missing_delivery_ts"
    )
)

fact_order = add_gold_metadata(fact_order, "silver_orders, silver_order_items, silver_order_payments, silver_order_reviews, dim_customer")

write_gold_table(fact_order, "fact_order")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 88, Finished, Available, Finished, False)

In [87]:
fact_order_saved = spark.table("fact_order")

fact_order_expected_count = (
    silver_orders.count()
)

fact_order_target_count = (
    fact_order_saved.count()
)

fact_order_distinct_grain_count = (
    fact_order_saved
    .select("order_id")
    .distinct()
    .count()
)

fact_order_duplicate_count = (
    fact_order_saved
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

fact_order_null_key_count = (
    fact_order_saved
    .filter(
        F.col("customer_key").isNull()
        | F.col("order_date_key").isNull()
    )
    .count()
)

fact_order_status = (
    "SUCCESS"
    if (
        fact_order_expected_count
        == fact_order_target_count
        == fact_order_distinct_grain_count
        and fact_order_duplicate_count == 0
        and fact_order_null_key_count == 0
    )
    else "FAILED"
)

record_fact_audit(
    fact_table="fact_order",
    grain_description="One row per order_id",
    source_tables=(
        "silver_orders, silver_order_items, "
        "silver_order_payments, silver_order_reviews"
    ),
    expected_row_count=fact_order_expected_count,
    target_row_count=fact_order_target_count,
    distinct_grain_count=(
        fact_order_distinct_grain_count
    ),
    duplicate_grain_count=(
        fact_order_duplicate_count
    ),
    null_dimension_key_count=(
        fact_order_null_key_count
    ),
    status=fact_order_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 89, Finished, Available, Finished, False)

# Order-Item Fact

Create one row per order and item sequence with customer, product, seller and purchase-date keys.

In [88]:
fact_order_item = (
    silver_order_items.alias("i")
    .join(
        silver_orders
        .select(
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_ts",
            "is_late_delivery"
        )
        .alias("o"),
        on="order_id",
        how="left"
    )
    .join(
        customer_lookup.alias("c"),
        on="customer_id",
        how="left"
    )
    .join(
        product_lookup.alias("p"),
        on="product_id",
        how="left"
    )
    .join(
        seller_lookup.alias("s"),
        on="seller_id",
        how="left"
    )
)

fact_order_item = (
    fact_order_item
    .withColumn(
        "order_date_key",
        F.date_format(
            F.to_date("order_purchase_ts"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "shipping_limit_date_key",
        F.when(
            F.col("shipping_limit_ts").isNotNull(),
            F.date_format(
                F.to_date("shipping_limit_ts"),
                "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "is_high_freight_ratio",
        F.when(
            F.col("freight_to_price_ratio") >= 0.5,
            True
        )
        .when(
            F.col("freight_to_price_ratio").isNotNull(),
            False
        )
    )
)

fact_order_item = (
    fact_order_item
    .select(
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        "order_date_key",
        "shipping_limit_date_key",
        "customer_id",
        "product_id",
        "seller_id",
        "order_status",
        "order_purchase_ts",
        "shipping_limit_ts",
        "is_late_delivery",
        "item_price",
        "freight_value",
        "item_total_value",
        "freight_to_price_ratio",
        "is_high_freight_ratio"
    )
)

fact_order_item = add_gold_metadata(
    fact_order_item,
    (
        "silver_order_items, "
        "silver_orders, "
        "dim_customer, "
        "dim_product, "
        "dim_seller"
    )
)

write_gold_table(
    fact_order_item,
    "fact_order_item"
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 90, Finished, Available, Finished, False)

In [89]:
fact_order_item_saved = spark.table(
    "fact_order_item"
)

fact_order_item_expected_count = (
    silver_order_items.count()
)

fact_order_item_target_count = (
    fact_order_item_saved.count()
)

fact_order_item_distinct_grain_count = (
    fact_order_item_saved
    .select(
        "order_id",
        "order_item_id"
    )
    .distinct()
    .count()
)

fact_order_item_duplicate_count = (
    fact_order_item_saved
    .groupBy(
        "order_id",
        "order_item_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

fact_order_item_null_key_count = (
    fact_order_item_saved
    .filter(
        F.col("customer_key").isNull()
        | F.col("product_key").isNull()
        | F.col("seller_key").isNull()
        | F.col("order_date_key").isNull()
    )
    .count()
)

fact_order_item_status = (
    "SUCCESS"
    if (
        fact_order_item_expected_count
        == fact_order_item_target_count
        == fact_order_item_distinct_grain_count
        and fact_order_item_duplicate_count == 0
        and fact_order_item_null_key_count == 0
    )
    else "FAILED"
)

record_fact_audit(
    fact_table="fact_order_item",
    grain_description=(
        "One row per order_id + order_item_id"
    ),
    source_tables=(
        "silver_order_items, silver_orders, "
        "dim_customer, dim_product, dim_seller"
    ),
    expected_row_count=(
        fact_order_item_expected_count
    ),
    target_row_count=(
        fact_order_item_target_count
    ),
    distinct_grain_count=(
        fact_order_item_distinct_grain_count
    ),
    duplicate_grain_count=(
        fact_order_item_duplicate_count
    ),
    null_dimension_key_count=(
        fact_order_item_null_key_count
    ),
    status=fact_order_item_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 91, Finished, Available, Finished, False)

# Payment Fact

Create one row per order and payment sequence with customer and purchase-date keys.

In [90]:
fact_payment = (
    silver_order_payments.alias("p")
    .join(
        silver_orders
        .select(
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_ts"
        )
        .alias("o"),
        on="order_id",
        how="left"
    )
    .join(
        customer_lookup.alias("c"),
        on="customer_id",
        how="left"
    )
)

fact_payment = (
    fact_payment
    .withColumn(
        "order_date_key",
        F.date_format(
            F.to_date("order_purchase_ts"),
            "yyyyMMdd"
        ).cast("int")
    )
)

fact_payment = (
    fact_payment
    .select(
        "order_id",
        "payment_sequence",
        "customer_key",
        "order_date_key",
        "customer_id",
        "order_status",
        "order_purchase_ts",
        "payment_type",
        "payment_installments",
        "uses_installments",
        "payment_installment_bucket",
        "payment_value"
    )
)

fact_payment = add_gold_metadata(
    fact_payment,
    (
        "silver_order_payments, "
        "silver_orders, "
        "dim_customer"
    )
)

write_gold_table(
    fact_payment,
    "fact_payment"
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 92, Finished, Available, Finished, False)

In [91]:
fact_payment_saved = spark.table(
    "fact_payment"
)

fact_payment_expected_count = (
    silver_order_payments.count()
)

fact_payment_target_count = (
    fact_payment_saved.count()
)

fact_payment_distinct_grain_count = (
    fact_payment_saved
    .select(
        "order_id",
        "payment_sequence"
    )
    .distinct()
    .count()
)

fact_payment_duplicate_count = (
    fact_payment_saved
    .groupBy(
        "order_id",
        "payment_sequence"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

fact_payment_null_key_count = (
    fact_payment_saved
    .filter(
        F.col("customer_key").isNull()
        | F.col("order_date_key").isNull()
    )
    .count()
)

fact_payment_status = (
    "SUCCESS"
    if (
        fact_payment_expected_count
        == fact_payment_target_count
        == fact_payment_distinct_grain_count
        and fact_payment_duplicate_count == 0
        and fact_payment_null_key_count == 0
    )
    else "FAILED"
)

record_fact_audit(
    fact_table="fact_payment",
    grain_description=(
        "One row per order_id + payment_sequence"
    ),
    source_tables=(
        "silver_order_payments, "
        "silver_orders, dim_customer"
    ),
    expected_row_count=(
        fact_payment_expected_count
    ),
    target_row_count=(
        fact_payment_target_count
    ),
    distinct_grain_count=(
        fact_payment_distinct_grain_count
    ),
    duplicate_grain_count=(
        fact_payment_duplicate_count
    ),
    null_dimension_key_count=(
        fact_payment_null_key_count
    ),
    status=fact_payment_status
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 93, Finished, Available, Finished, False)

In [92]:
failed_fact_loads = [
    record
    for record in gold_fact_audit_records
    if record["load_status"] != "SUCCESS"
]

if failed_fact_loads:
    raise RuntimeError(
        f"Gold fact validation failed: "
        f"{failed_fact_loads}"
    )

print("All three Gold facts validated successfully.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 94, Finished, Available, Finished, False)

All three Gold facts validated successfully.


In [93]:
gold_fact_audit_schema = StructType([
    StructField("gold_run_id", StringType(), False),
    StructField("fact_table", StringType(), False),
    StructField("grain_description", StringType(), False),
    StructField("source_tables", StringType(), False),
    StructField("expected_row_count", LongType(), False),
    StructField("target_row_count", LongType(), False),
    StructField("distinct_grain_count", LongType(), False),
    StructField("duplicate_grain_count", LongType(), False),
    StructField("null_dimension_key_count", LongType(), False),
    StructField("row_count_difference", LongType(), False),
    StructField("load_status", StringType(), False),
    StructField("processed_at_utc", TimestampType(), False)
])

gold_fact_audit_df = spark.createDataFrame(gold_fact_audit_records, schema=gold_fact_audit_schema)

display(gold_fact_audit_df.orderBy("fact_table"))

gold_fact_audit_df.write.mode("append").format("delta").saveAsTable("audit_gold_fact_load")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 95, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28b0156d-0d75-4b5e-9730-d96107639e86)

In [97]:
GOLD_RELATIONSHIPS = [
    {
        "relationship_id": "GREL001",
        "relationship_name": "Fact Order to Customer",
        "fact_table": "fact_order",
        "fact_key": "customer_key",
        "dimension_table": "dim_customer",
        "dimension_key": "customer_key"
    },
    {
        "relationship_id": "GREL002",
        "relationship_name": "Fact Order to Date",
        "fact_table": "fact_order",
        "fact_key": "order_date_key",
        "dimension_table": "dim_date",
        "dimension_key": "date_key"
    },
    {
        "relationship_id": "GREL003",
        "relationship_name": "Fact Item to Customer",
        "fact_table": "fact_order_item",
        "fact_key": "customer_key",
        "dimension_table": "dim_customer",
        "dimension_key": "customer_key"
    },
    {
        "relationship_id": "GREL004",
        "relationship_name": "Fact Item to Product",
        "fact_table": "fact_order_item",
        "fact_key": "product_key",
        "dimension_table": "dim_product",
        "dimension_key": "product_key"
    },
    {
        "relationship_id": "GREL005",
        "relationship_name": "Fact Item to Seller",
        "fact_table": "fact_order_item",
        "fact_key": "seller_key",
        "dimension_table": "dim_seller",
        "dimension_key": "seller_key"
    },
    {
        "relationship_id": "GREL006",
        "relationship_name": "Fact Item to Date",
        "fact_table": "fact_order_item",
        "fact_key": "order_date_key",
        "dimension_table": "dim_date",
        "dimension_key": "date_key"
    },
    {
        "relationship_id": "GREL007",
        "relationship_name": "Fact Payment to Customer",
        "fact_table": "fact_payment",
        "fact_key": "customer_key",
        "dimension_table": "dim_customer",
        "dimension_key": "customer_key"
    },
    {
        "relationship_id": "GREL008",
        "relationship_name": "Fact Payment to Date",
        "fact_table": "fact_payment",
        "fact_key": "order_date_key",
        "dimension_table": "dim_date",
        "dimension_key": "date_key"
    }
]

gold_relationship_records = []

for relationship in GOLD_RELATIONSHIPS:

    fact_keys = (
        spark.table(relationship["fact_table"])
        .select(F.col(relationship["fact_key"]).alias("relationship_key"))
        .filter(F.col("relationship_key").isNotNull())
    )

    dimension_keys = (
        spark.table(relationship["dimension_table"])
        .select(F.col(relationship["dimension_key"]).alias("relationship_key"))
        .filter(F.col("relationship_key").isNotNull())
        .dropDuplicates()
    )

    evaluated_fact_rows = fact_keys.count()

    orphan_rows = fact_keys.join(dimension_keys, on="relationship_key", how="left_anti").count()

    gold_relationship_records.append({
        "gold_run_id": GOLD_RUN_ID,
        "relationship_id": relationship["relationship_id"],
        "relationship_name": relationship["relationship_name"],
        "fact_table": relationship["fact_table"],
        "fact_key": relationship["fact_key"],
        "dimension_table": relationship["dimension_table"],
        "dimension_key": relationship["dimension_key"],
        "evaluated_fact_row_count": int(evaluated_fact_rows),
        "orphan_row_count": int(orphan_rows),
        "orphan_percentage": (
            builtins.round(orphan_rows / evaluated_fact_rows * 100, 4)
            if evaluated_fact_rows > 0
            else 0.0
        ),
        "status": (
            "PASS"
            if orphan_rows == 0
            else "FAIL"
        ),
        "tested_at_utc": datetime.now(timezone.utc)
    })

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 99, Finished, Available, Finished, False)

In [98]:
gold_relationship_df = spark.createDataFrame(gold_relationship_records)
display(gold_relationship_df.orderBy("relationship_id"))
gold_relationship_df.write.mode("append").format("delta").saveAsTable("audit_gold_relationship")

failed_gold_relationships = gold_relationship_df.filter(F.col("status") == "FAIL").count()

if failed_gold_relationships > 0:
    raise RuntimeError("Gold fact-to-dimension relationship validation failed.")

print("All Gold relationships passed.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 100, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0181382f-f15c-4b94-8afc-9928bf5720cf)

All Gold relationships passed.


In [99]:
# Calculate reconciliation values

silver_item_totals = (
    silver_order_items
    .agg(
        F.sum("item_price").alias("silver_gmv"),
        F.sum("freight_value").alias("silver_freight"),
        F.sum("item_total_value").alias("silver_total_value")
    )
    .collect()[0]
)

gold_item_totals = (
    fact_order_item_saved
    .agg(
        F.sum("item_price").alias("gold_gmv"),
        F.sum("freight_value").alias("gold_freight"),
        F.sum("item_total_value").alias("gold_total_value")
    )
    .collect()[0]
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 101, Finished, Available, Finished, False)

In [100]:
silver_payment_total = (
    silver_order_payments
    .agg(F.sum("payment_value").alias("silver_payment_value"))
    .collect()[0]["silver_payment_value"]
)

gold_payment_total = (
    fact_payment_saved
    .agg(F.sum("payment_value").alias("gold_payment_value"))
    .collect()[0]["gold_payment_value"]
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 102, Finished, Available, Finished, False)

In [101]:
gold_order_totals = (
    fact_order_saved
    .agg(F.sum("order_gmv").alias("fact_order_gmv"),
        F.sum("order_freight_value").alias("fact_order_freight"),
        F.sum("order_total_value").alias("fact_order_total_value"),
        F.sum("order_payment_value").alias("fact_order_payment_value")
    )
    .collect()[0]
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 103, Finished, Available, Finished, False)

In [102]:
def decimal_difference(value_1, value_2) -> float:
    return builtins.round(float(value_1 or 0) - float(value_2 or 0), 2)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 104, Finished, Available, Finished, False)

In [103]:
gold_reconciliation_records = [
    {
        "gold_run_id": GOLD_RUN_ID,
        "reconciliation_id": "GREC001",
        "metric_name": "GMV: Silver Items vs Gold Item Fact",
        "source_value": float(silver_item_totals["silver_gmv"] or 0),
        "target_value": float(gold_item_totals["gold_gmv"] or 0),
        "difference": decimal_difference(silver_item_totals["silver_gmv"], gold_item_totals["gold_gmv"]),
        "tolerance": 0.01
    },
    {
        "gold_run_id": GOLD_RUN_ID,
        "reconciliation_id": "GREC002",
        "metric_name": "Freight: Silver Items vs Gold Item Fact",
        "source_value": float(silver_item_totals["silver_freight"] or 0),
        "target_value": float(gold_item_totals["gold_freight"] or 0),
        "difference": decimal_difference(silver_item_totals["silver_freight"], gold_item_totals["gold_freight"]),
        "tolerance": 0.01
    },
    {
        "gold_run_id": GOLD_RUN_ID,
        "reconciliation_id": "GREC003",
        "metric_name": "Payment: Silver Payments vs Gold Payment Fact",
        "source_value": float(silver_payment_total or 0),
        "target_value": float(gold_payment_total or 0),
        "difference": decimal_difference(silver_payment_total, gold_payment_total),
        "tolerance": 0.01
    },
    {
        "gold_run_id": GOLD_RUN_ID,
        "reconciliation_id": "GREC004",
        "metric_name": "GMV: Item Fact vs Order Fact",
        "source_value": float(gold_item_totals["gold_gmv"] or 0),
        "target_value": float(gold_order_totals["fact_order_gmv"] or 0),
        "difference": decimal_difference(gold_item_totals["gold_gmv"], gold_order_totals["fact_order_gmv"]),
        "tolerance": 0.01
    },
    {
        "gold_run_id": GOLD_RUN_ID,
        "reconciliation_id": "GREC005",
        "metric_name": ("Payment: Payment Fact vs Order Fact"),
        "source_value": float(gold_payment_total or 0),
        "target_value": float(gold_order_totals["fact_order_payment_value"] or 0),
        "difference": decimal_difference(gold_payment_total, gold_order_totals["fact_order_payment_value"]),
        "tolerance": 0.01
    }
]

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 105, Finished, Available, Finished, False)

In [104]:
for record in gold_reconciliation_records:
    record["status"] = "PASS" if abs(record["difference"]) <= record["tolerance"] else "FAIL"
    record["tested_at_utc"] = datetime.now(timezone.utc)

gold_reconciliation_df = spark.createDataFrame(gold_reconciliation_records)

display(gold_reconciliation_df.orderBy("reconciliation_id"))

gold_reconciliation_df.write.mode("append").format("delta").saveAsTable("audit_gold_reconciliation")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 106, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdae2ee6-7e07-4d0e-b048-687517fe634a)

In [105]:
failed_reconciliations = gold_reconciliation_df.filter(F.col("status") == "FAIL").count()
    
if failed_reconciliations > 0:
    raise RuntimeError("Gold financial reconciliation failed.")

print("All Gold reconciliations passed.")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 107, Finished, Available, Finished, False)

All Gold reconciliations passed.


In [106]:
EXPECTED_GOLD_TABLES = {
    "dim_date",
    "dim_customer",
    "dim_product",
    "dim_seller",
    "fact_order",
    "fact_order_item",
    "fact_payment"
}

missing_gold_tables = {
    table_name
    for table_name in EXPECTED_GOLD_TABLES
    if not spark.catalog.tableExists(
        table_name
    )
}

if missing_gold_tables:
    raise RuntimeError(f"Missing Gold tables: {missing_gold_tables}")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 108, Finished, Available, Finished, False)

In [108]:
GOLD_COMPLETED_AT = datetime.now(timezone.utc)

GOLD_DURATION_SECONDS = builtins.round(time.perf_counter() - GOLD_START_PREF, 2)

gold_run_history_record = [{
    "gold_run_id": GOLD_RUN_ID,
    "started_at_utc": GOLD_STARTED_AT,
    "completed_at_utc": GOLD_COMPLETED_AT,
    "duration_seconds": GOLD_DURATION_SECONDS,
    "load_type": LOAD_TYPE,
    "dimension_table_count": 4,
    "fact_table_count": 3,
    "successful_dimension_count": sum(
        record["load_status"] == "SUCCESS"
        for record in gold_dimension_audit_records
    ),
    "successful_fact_count": sum(
        record["load_status"] == "SUCCESS"
        for record in gold_fact_audit_records
    ),
    "failed_relationship_count": int(failed_gold_relationships),
    "failed_reconciliation_count": int(failed_reconciliations),
    "status": (
        "SUCCESS"
        if not failed_fact_loads and failed_gold_relationships == 0 and failed_reconciliations == 0
        else "FAILED"
    )
}]

gold_run_history_df = spark.createDataFrame(gold_run_history_record)

display(gold_run_history_df)
gold_run_history_df.write.mode("append").format("delta").saveAsTable("audit_gold_run_history")

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 110, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a41dfd5c-38f5-46d7-b005-8a5b6b2adcf7)

In [109]:
## Gold facts

display(
    spark.table("fact_order")
    .select(
        "order_id",
        "customer_key",
        "order_date_key",
        "order_status",
        "order_gmv",
        "order_freight_value",
        "order_total_value",
        "order_payment_value",
        "item_count",
        "review_count",
        "average_review_score",
        "delivery_days",
        "is_late_delivery"
    )
    .orderBy(F.col("order_gmv").desc())
    .limit(20)
)

## Order-item fact

display(
    spark.table("fact_order_item")
    .select(
        "order_id",
        "order_item_id",
        "customer_key",
        "product_key",
        "seller_key",
        "order_date_key",
        "item_price",
        "freight_value",
        "item_total_value",
        "freight_to_price_ratio"
    )
    .limit(20)
)

## Payment fact

display(
    spark.table("fact_payment")
    .select(
        "order_id",
        "payment_sequence",
        "customer_key",
        "order_date_key",
        "payment_type",
        "payment_installments",
        "payment_installment_bucket",
        "payment_value"
    )
    .limit(20)
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cfd7c93e-1925-4a25-a316-b925fd4f2cc3)

SynapseWidget(Synapse.DataFrame, 4f3f826e-c9c3-4567-baf3-67c630cb0faa)

SynapseWidget(Synapse.DataFrame, d36add06-f53c-4f67-80cf-c63fb8658b46)

In [110]:
display(
    spark.table("audit_gold_fact_load")
    .filter(
        F.col("gold_run_id") == GOLD_RUN_ID
    )
    .orderBy("fact_table")
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 112, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9f903c7a-aae2-4274-91ab-9feabee22c21)

In [111]:
display(
    spark.table("audit_gold_relationship")
    .filter(
        F.col("gold_run_id") == GOLD_RUN_ID
    )
    .orderBy("relationship_id")
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 113, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eaba7e0b-1d97-454d-ba2a-c5e45257a5fb)

In [112]:
display(
    spark.table("audit_gold_reconciliation")
    .filter(
        F.col("gold_run_id") == GOLD_RUN_ID
    )
    .orderBy("reconciliation_id")
)

StatementMeta(, 92e2f5c0-36e9-45a1-a484-d60c41809eb2, 114, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 782c2f5a-fcdf-43ed-b0e4-0cf6a42dd427)